# Sentiment Analysis on Twitter Dataset
This notebook demonstrates how to implement sentiment analysis using Naive Bayes classifier on the Kaggle Twitter Entity Sentiment Analysis dataset.

In [ ]:
# Install NLTK resources (if not already installed)
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Load libraries
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Load the datasets
train_df = pd.read_csv('twitter_training.csv', header=None, names=['Tweet_ID', 'Entity', 'Sentiment', 'Text'])
val_df = pd.read_csv('twitter_validation.csv', header=None, names=['Tweet_ID', 'Entity', 'Sentiment', 'Text'])

In [ ]:
# Filter to Positive and Negative only
train_df = train_df[train_df['Sentiment'].isin(['Positive', 'Negative'])]
val_df = val_df[val_df['Sentiment'].isin(['Positive', 'Negative'])]

# Map sentiments to binary labels
sentiment_mapping = {'Positive': 1, 'Negative': 0}
train_df['Sentiment'] = train_df['Sentiment'].map(sentiment_mapping)
val_df['Sentiment'] = val_df['Sentiment'].map(sentiment_mapping)

In [ ]:
# Preprocessing function
def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', token.lower()) for token in tokens]
    tokens = [token for token in tokens if token not in stopwords.words('english')]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    return ' '.join(tokens)

# Apply preprocessing
train_df['Processed_Text'] = train_df['Text'].apply(preprocess_text)
val_df['Processed_Text'] = val_df['Text'].apply(preprocess_text)

In [ ]:
# Vectorization using Bag of Words
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_df['Processed_Text'])
X_val = vectorizer.transform(val_df['Processed_Text'])

In [ ]:
# Train and evaluate Naive Bayes classifier
clf = MultinomialNB()
clf.fit(X_train, train_df['Sentiment'])
y_pred = clf.predict(X_val)

accuracy = accuracy_score(val_df['Sentiment'], y_pred)
report = classification_report(val_df['Sentiment'], y_pred)

print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(report)